# Study 934 — Lump Sum vs DCA — the teardown

Terminal-wealth race over every start month: win rate with a Wilson interval, the overlap-corrected HAC *t* on the mean gap, a non-overlapping check, a 12-month block bootstrap, the **exposure-matched control** that decides what the gap is made of, the dispersion read, the conditional cuts, era and decade cuts, a long-history extension, the cost/ticket/tranche sweeps, the bond-heavy variant and the two-sided synthetic control.

**Design.** $1, twelve months, both arms valued on the same terminal date. Purchases are decided at a month-end close and executed at the next trading day's close — one lag, applied identically to the lump-sum buy and to all twelve tranches. Uninvested DCA balance accrues **BIL total return**. Costs one-way × NAV; no shorting anywhere, so no borrow leg.

Real numbers frozen from `docs/results.md` (SPY fingerprint `edef65f148a6`, IEF `9803a2a6157d`), as-of 2026-06-30.

In [1]:
R = {'start': '2007-05-30', 'end': '2026-06-30', 'n_days': 4802, 'n_win': 217, 'fp_spy': 'edef65f148a6', 'fp_ief': '9803a2a6157d', 'first_start': '2007-06-01', 'last_end': '2026-06-01', 'win': 76.0, 'win_lo': 69.9, 'win_hi': 81.2, 'mean_gap': 5.05, 'median_gap': 6.09, 'sd_gap': 9.93, 'p05': -10.52, 'p95': 18.73, 'worst_gap': -30.65, 'best_gap': 43.93, 't_hac': 3.19, 't_nonoverlap': 2.18, 'boot_lo': 1.57, 'boot_hi': 7.9, 'boot_neg': 0.2, 'lump_mean': 12.35, 'dca_mean': 7.29, 'lump_worst': -45.85, 'dca_worst': -36.13, 'sd_lump': 0.1701, 'sd_dca': 0.0999, 'disp_ratio': 0.587, 'cheap_n': 61, 'cheap_win': 93.4, 'cheap_gap': 10.1, 'cheap_t': 9.66, 'cheap_ret': 22.1, 'mid_n': 60, 'mid_win': 80.0, 'mid_gap': 5.64, 'mid_t': 5.22, 'mid_ret': 13.8, 'str_n': 60, 'str_win': 73.3, 'str_gap': 3.43, 'str_t': 2.38, 'str_ret': 8.5, 'dd_n': 59, 'dd_win': 72.9, 'dd_gap': 5.26, 'dd_med': 7.48, 'dd_t': 1.5, 'dd_ret': 13.8, 'hi_n': 158, 'hi_win': 77.2, 'hi_gap': 4.98, 'hi_t': 3.63, 'hi_ret': 11.8, 'era_e_n': 115, 'era_e_win': 70.4, 'era_e_gap': 4.09, 'era_e_t': 1.65, 'era_l_n': 102, 'era_l_win': 82.4, 'era_l_gap': 6.14, 'era_l_t': 3.39, 'long_start': '2000-01-03', 'long_n': 305, 'long_win': 74.4, 'long_gap': 4.47, 'long_t': 3.25, 'long_t_no': 2.36, 'long_e_gap': 0.67, 'long_e_t': 0.26, 'long_l_gap': 6.9, 'long_l_t': 6.01, 'dec00_n': 119, 'dec00_win': 60.5, 'dec00_gap': 0.67, 'dec00_t': 0.26, 'dec00_ret': 1.6, 'dec10_n': 120, 'dec10_win': 85.0, 'dec10_gap': 6.1, 'dec10_t': 6.61, 'dec10_ret': 13.5, 'dec20_n': 66, 'dec20_win': 80.3, 'dec20_gap': 8.36, 'dec20_t': 3.06, 'dec20_ret': 17.6, 'long_em_gap': 0.03, 'long_em_t': 0.06, 'em_w': 54.2, 'em_gap': -0.04, 'em_win': 53.5, 'em_t': -0.08, 'em_t_no': -0.06, 'em_lo': -1.18, 'em_hi': 0.98, 'em_sd': 0.0926, 'dm_w': 58.4, 'dm_gap': 0.43, 'dm_t': 0.78, 'dm_lo': -0.8, 'dm_hi': 1.48, 'rd_lump': 0.651, 'rd_dca': 0.608, 'xs_lump': 11.11, 'xs_dca': 6.06, 'ief_em_gap': 0.08, 'ief_em_t': 0.47, 'ief_em_lo': -0.26, 'ief_em_hi': 0.42, 'cost0': 5.05, 'cost25': 5.04, 'tick0': 5.05, 'tick1': 5.16, 'tick5': 5.6, 'tick10': 6.15, 'zero_cash_gap': 5.64, 'zero_cash_t': 3.57, 'tr3_gap': 0.81, 'tr6_gap': 2.18, 'tr12_gap': 5.05, 'tr24_gap': 11.61, 'tr3_win': 65.0, 'tr6_win': 70.9, 'tr12_win': 76.0, 'tr24_win': 86.8, 'tr3_disp': 0.727, 'tr6_disp': 0.639, 'tr12_disp': 0.587, 'tr24_disp': 0.497, 'ief_win': 59.0, 'ief_gap': 1.02, 'ief_t': 1.45, 'ief_lo': -0.36, 'ief_hi': 2.29, 'ief_disp': 0.628, 'syn_pl_gap': 3.74, 'syn_pl_win': 62.7, 'syn_pl_seeds': '12/12', 'syn_nl_gap': 0.13, 'syn_nl_win': 46.6, 'syn_nl_seeds': '6/12', 'syn_fa_gap': -3.15, 'syn_fa_win': 32.0, 'syn_fa_seeds': '0/12'}

## 1. Headline and its inference

Monthly starts with twelve-month horizons overlap by up to eleven months, so the naive *t* is meaningless. Three independent corrections:

In [2]:
print(f"n windows {R['n_win']}   {R['first_start']} -> {R['last_end']}")
print(f"win rate      : {R['win']:.1f}%  Wilson 95% CI [{R['win_lo']:.1f}%, {R['win_hi']:.1f}%]")
print(f"mean gap      : {R['mean_gap']:+.2f}c   median {R['median_gap']:+.2f}c   sd {R['sd_gap']:.2f}c")
print(f"HAC t (12 lag): {R['t_hac']:+.2f}")
print(f"non-overlap t : {R['t_nonoverlap']:+.2f}   (every 12th start, averaged over all 12 phases)")
print(f"boot 95% CI   : [{R['boot_lo']:+.2f}, {R['boot_hi']:+.2f}]c   share<0 {R['boot_neg']:.1f}%")

n windows 217   2007-06-01 -> 2026-06-01
win rate      : 76.0%  Wilson 95% CI [69.9%, 81.2%]
mean gap      : +5.05c   median +6.09c   sd 9.93c
HAC t (12 lag): +3.19
non-overlap t : +2.18   (every 12th start, averaged over all 12 phases)
boot 95% CI   : [+1.57, +7.90]c   share<0 0.2%


> 💡 **In plain words** — three different ways of admitting the windows share tape, and all three still say the lump sum wins by about five cents on the dollar. Whether those cents are *timing* is section 1b.

## 1b. Exposure or timing? — the control that sets the Tradability stamp

A DCA schedule is not only later into the market, it is **less in** it. Tranche *j* is invested for (n−j)/n of the window, so its time-weighted exposure is the analytic **(n+1)/2n = 13/24 = 54.2%** — nothing fitted, nothing peeked at. A raw terminal-wealth race therefore pits a full-beta portfolio against a half-beta one, and any positive premium hands the lump sum a win it did not earn by timing.

The control: race DCA against a **static** portfolio holding that same 54.2% in the asset and the rest in the same BIL leg for the whole window. Both arms are then read excess of the **same** cash leg — never one raw against one excess.

In [3]:
print(f"headline  lump vs DCA       : {R['mean_gap']:+.2f}c  "
      f"HAC t {R['t_hac']:+.2f}  CI [{R['boot_lo']:+.2f}, {R['boot_hi']:+.2f}]")
print(f"matched   {R['em_w']:.1f}% static vs DCA : {R['em_gap']:+.2f}c  "
      f"HAC t {R['em_t']:+.2f}  CI [{R['em_lo']:+.2f}, {R['em_hi']:+.2f}]  "
      f"(non-overlap t {R['em_t_no']:+.2f}, win {R['em_win']:.1f}%)")
print(f"          {R['dm_w']:.1f}% static vs DCA : {R['dm_gap']:+.2f}c  "
      f"HAC t {R['dm_t']:+.2f}  CI [{R['dm_lo']:+.2f}, {R['dm_hi']:+.2f}]  "
       '(dispersion-matched weight = IN-SAMPLE fit, sensitivity only)')
print()
print(f"reward per unit dispersion, both excess of the same cash leg:")
print(f"   lump {R['xs_lump']:+.2f}% -> {R['rd_lump']:.3f}   "
      f"DCA {R['xs_dca']:+.2f}% -> {R['rd_dca']:.3f}")
print(f"long tape ({R['long_start'][:4]}-2026), same control: "
      f"{R['long_em_gap']:+.2f}c (t {R['long_em_t']:+.2f})")

headline  lump vs DCA       : +5.05c  HAC t +3.19  CI [+1.57, +7.90]
matched   54.2% static vs DCA : -0.04c  HAC t -0.08  CI [-1.18, +0.98]  (non-overlap t -0.06, win 53.5%)
          58.4% static vs DCA : +0.43c  HAC t +0.78  CI [-0.80, +1.48]  (dispersion-matched weight = IN-SAMPLE fit, sensitivity only)

reward per unit dispersion, both excess of the same cash leg:
   lump +11.11% -> 0.651   DCA +6.06% -> 0.608
long tape (2000-2026), same control: +0.03c (t +0.06)


> 💡 **In plain words** — every cent of the headline is the extra beta. Own DCA's average weight for twelve months and you land where DCA lands, with the same dispersion, without the schedule. That is the whole finding, and it is why the second stamp is red.

## 2. Dispersion — the risk claim, stated separately from the return claim

In [4]:
print(f"terminal wealth  lump: mean {R['lump_mean']:+.2f}%  sd {R['sd_lump']:.4f}  worst {R['lump_worst']:.2f}%")
print(f"                 DCA : mean {R['dca_mean']:+.2f}%  sd {R['sd_dca']:.4f}  worst {R['dca_worst']:.2f}%")
print(f"dispersion ratio (DCA/lump) = {R['disp_ratio']:.3f}")
print(f"gap distribution: p5 {R['p05']:+.2f}c   p95 {R['p95']:+.2f}c   "
      f"worst {R['worst_gap']:+.2f}c   best {R['best_gap']:+.2f}c")

terminal wealth  lump: mean +12.35%  sd 0.1701  worst -45.85%
                 DCA : mean +7.29%  sd 0.0999  worst -36.13%
dispersion ratio (DCA/lump) = 0.587
gap distribution: p5 -10.52c   p95 +18.73c   worst -30.65c   best +43.93c


The dispersion cut is real and mechanical: DCA's average equity exposure over the window is ~54% of the lump sum's, so its variance is roughly a third and its SD roughly six tenths. Nothing about *prices* is being improved.

## 3. Conditional cuts — hindsight terciles, PROXY valuation

`stretch0` = level ÷ trailing 3-year mean at the start date. It is a **price proxy** for expensive, not CAPE (no earnings data enters this study), and the terciles are cut in-sample, so this answers "did such a state exist?" and not "could you trade it?". The trailing mean needs a three-year runway inside the sample, so the terciles cover 181 of the 217 starts; the drawdown cut uses all 217.

In [5]:
rows = [('cheap tercile', R['cheap_n'], R['cheap_win'], R['cheap_gap'], R['cheap_t'], R['cheap_ret']),
        ('middle tercile', R['mid_n'], R['mid_win'], R['mid_gap'], R['mid_t'], R['mid_ret']),
        ('stretched tercile', R['str_n'], R['str_win'], R['str_gap'], R['str_t'], R['str_ret']),
        ('start >=10% below high', R['dd_n'], R['dd_win'], R['dd_gap'], R['dd_t'], R['dd_ret']),
        ('start near the highs', R['hi_n'], R['hi_win'], R['hi_gap'], R['hi_t'], R['hi_ret'])]
print(f"{'start state':24s} {'n':>4s} {'win%':>7s} {'gap':>8s} {'t':>7s} {'12m ret':>9s}")
for name, n, w, g, t, r in rows:
    print(f"{name:24s} {n:4d} {w:6.1f}% {g:+7.2f}c {t:+7.2f} {r:+8.1f}%")
print('\nno cut crosses zero; the weakest t is the drawdown cut (n=59)')

start state                 n    win%      gap       t   12m ret
cheap tercile              61   93.4%  +10.10c   +9.66    +22.1%
middle tercile             60   80.0%   +5.64c   +5.22    +13.8%
stretched tercile          60   73.3%   +3.43c   +2.38     +8.5%
start >=10% below high     59   72.9%   +5.26c   +1.50    +13.8%
start near the highs      158   77.2%   +4.98c   +3.63    +11.8%

no cut crosses zero; the weakest t is the drawdown cut (n=59)


> 💡 **In plain words** — the one place the advice should shine, starting from a drawdown, is the one place the evidence is thinnest (*t* = +1.50) — and even there the sign is still against it.

## 4. Era cuts, decades, and the long-history extension

BIL's 2007 inception gates the honest cash leg. The extension backwards runs under a **0% cash ASSUMPTION**, and is floored at a **pinned 2000-01-03** rather than at whatever SPY history the *shared* `studies/_cache` happens to hold — other studies re-pull the same tickers with their own start dates, and a robustness number that moves with someone else's fetch is not a robustness number. (An earlier draft of this study quoted a 1993-start run; it no longer reproduces, and the conclusion it supported was wrong. The decade table is what replaced it.)

In [6]:
print(f"2007-06..2016-12 (n={R['era_e_n']}): win {R['era_e_win']:.1f}%  gap {R['era_e_gap']:+.2f}c  t={R['era_e_t']:+.2f}")
print(f"2017-01..2025-06 (n={R['era_l_n']}): win {R['era_l_win']:.1f}%  gap {R['era_l_gap']:+.2f}c  t={R['era_l_t']:+.2f}")
print()
print(f"long history {R['long_start'][:4]}-2026 (n={R['long_n']}, 0% cash ASSUMPTION, pinned start): "
      f"win {R['long_win']:.1f}%  gap {R['long_gap']:+.2f}c  HAC t {R['long_t']:+.2f}  "
      f"non-overlap t {R['long_t_no']:+.2f}")
for tag, n, w, g, t, r in [('2000s', R['dec00_n'], R['dec00_win'], R['dec00_gap'], R['dec00_t'], R['dec00_ret']),
                           ('2010s', R['dec10_n'], R['dec10_win'], R['dec10_gap'], R['dec10_t'], R['dec10_ret']),
                           ('2020s', R['dec20_n'], R['dec20_win'], R['dec20_gap'], R['dec20_t'], R['dec20_ret'])]:
    print(f"   {tag}: n={n:3d}  win {w:5.1f}%  gap {g:+6.2f}c  t={t:+5.2f}  mean 12m SPY {r:+5.1f}%")
print(f"   exposure-matched on the long tape: {R['long_em_gap']:+.2f}c (t {R['long_em_t']:+.2f})")

2007-06..2016-12 (n=115): win 70.4%  gap +4.09c  t=+1.65
2017-01..2025-06 (n=102): win 82.4%  gap +6.14c  t=+3.39

long history 2000-2026 (n=305, 0% cash ASSUMPTION, pinned start): win 74.4%  gap +4.47c  HAC t +3.25  non-overlap t +2.36
   2000s: n=119  win  60.5%  gap  +0.67c  t=+0.26  mean 12m SPY  +1.6%
   2010s: n=120  win  85.0%  gap  +6.10c  t=+6.61  mean 12m SPY +13.5%
   2020s: n= 66  win  80.3%  gap  +8.36c  t=+3.06  mean 12m SPY +17.6%
   exposure-matched on the long tape: +0.03c (t +0.06)


> 💡 **In plain words** — across the lost decade, the decade whose equities paid **+1.6%** a year over twelve months, the lump sum's advantage is **+0.67c** with *t* = **+0.26**: gone, not merely smaller. The advantage is the size of the premium that showed up — exactly what the exposure control in section 1b predicts.

## 5. Costs — the one that cancels and the one that does not

Both arms put the same $1 to work, so a **proportional** one-way cost is charged on the same notional in each and cancels in the difference. A **fixed ticket** does not: DCA pays it twelve times. Ticket amounts and the $10,000 windfall are ASSUMPTIONS, swept rather than asserted.

In [7]:
print(f"proportional: 0 bps {R['cost0']:+.2f}c  ->  25 bps {R['cost25']:+.2f}c   (cancels, by construction)")
print(f"fixed ticket on a $10,000 windfall:")
for tk, g in [(0, R['tick0']), (1, R['tick1']), (5, R['tick5']), (10, R['tick10'])]:
    print(f"   ${tk:2d}/trade -> mean gap {g:+.2f}c")
print(f"\ncash-leg assumption: 0% cash gives {R['zero_cash_gap']:+.2f}c (t={R['zero_cash_t']:+.2f}) "
      f"vs {R['mean_gap']:+.2f}c crediting the real BIL path")

proportional: 0 bps +5.05c  ->  25 bps +5.04c   (cancels, by construction)
fixed ticket on a $10,000 windfall:
   $ 0/trade -> mean gap +5.05c
   $ 1/trade -> mean gap +5.16c
   $ 5/trade -> mean gap +5.60c
   $10/trade -> mean gap +6.15c

cash-leg assumption: 0% cash gives +5.64c (t=+3.57) vs +5.05c crediting the real BIL path


## 6. Tranche length and the bond-heavy variant

In [8]:
print('tranches   win%     gap    dispersion ratio')
for k, w, g, d in [(3, R['tr3_win'], R['tr3_gap'], R['tr3_disp']),
                   (6, R['tr6_win'], R['tr6_gap'], R['tr6_disp']),
                   (12, R['tr12_win'], R['tr12_gap'], R['tr12_disp']),
                   (24, R['tr24_win'], R['tr24_gap'], R['tr24_disp'])]:
    print(f'   {k:2d}     {w:5.1f}%  {g:+6.2f}c        {d:.3f}')
print()
print(f"IEF (7-10y Treasuries): win {R['ief_win']:.1f}%  gap {R['ief_gap']:+.2f}c  "
      f"t={R['ief_t']:+.2f}  boot CI [{R['ief_lo']:+.2f}, {R['ief_hi']:+.2f}] -> includes zero")
print(f"   exposure-matched on IEF: {R['ief_em_gap']:+.2f}c (t {R['ief_em_t']:+.2f}, "
      f"CI [{R['ief_em_lo']:+.2f}, {R['ief_em_hi']:+.2f}])")
print()
print('tranche count only moves average exposure: (n+1)/2n = 66.7 / 58.3 / 54.2 / 52.1%')

tranches   win%     gap    dispersion ratio
    3      65.0%   +0.81c        0.727
    6      70.9%   +2.18c        0.639
   12      76.0%   +5.05c        0.587
   24      86.8%  +11.61c        0.497

IEF (7-10y Treasuries): win 59.0%  gap +1.02c  t=+1.45  boot CI [-0.36, +2.29] -> includes zero
   exposure-matched on IEF: +0.08c (t +0.47, CI [-0.26, +0.42])

tranche count only moves average exposure: (n+1)/2n = 66.7 / 58.3 / 54.2 / 52.1%


> 💡 **In plain words** — the bond result is the honest boundary of the finding: the lump sum's edge *is* the premium of the asset you are buying, so where the premium is thin the edge is statistically invisible. And the tranche sweep is the same fact along a second axis — return and dispersion move together at every point on that line because the only thing the tranche count changes is the average weight.

## 7. The two-sided synthetic control (live, offline)

A harness that only ever crowns the lump sum would look 'validated' by a rising tape. The control runs three planted worlds and must get all three right. One 25-year path of a 16%-vol asset carries a ~3 pp standard error on its own drift, so the control is read across seeds — a single draw genuinely can land the wrong side of a 7 pp planted premium.

In [9]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from lump_vs_dca import data, strategy as st

def small(signal_strength, seed):
    return data.synthetic_daily(n_years=15, signal_strength=signal_strength, seed=seed)

for ss in (1.0, 0.0, -1.0):
    c = st.synthetic_control(ss, seeds=range(934, 940), synth=small)
    print(f"ss={ss:+.1f}: mean gap {c['mean_gap_cents']:+6.2f}c (sd {c['sd_gap_cents']:.2f}), "
          f"mean win rate {c['mean_win_rate']:.1%}, lump wins on {c['frac_seeds_lump_wins']:.0%} of seeds")

ss=+1.0: mean gap  +4.45c (sd 2.82), mean win rate 66.0%, lump wins on 100% of seeds


ss=+0.0: mean gap  +0.76c (sd 2.56), mean win rate 47.8%, lump wins on 67% of seeds


ss=-1.0: mean gap  -2.58c (sd 2.32), mean win rate 32.6%, lump wins on 17% of seeds


The 12-seed version quoted in `docs/results.md` reads **+3.74c** (12/12 seeds) on the planted world, **+0.13c** (6/12) on the null and **-3.15c** (0/12) on the falling tape — the harness crowns DCA when DCA deserves it.

## Verdict

- **Signal — Real.** Win rate **76.0%** (Wilson CI [69.9%, 81.2%], clear of 50%), mean gap **+5.05c/$1**, HAC *t* **+3.19**, non-overlapping *t* **+2.18**, bootstrap CI **[+1.57, +7.90]**. Sign-stable across both eras (+4.09c / +6.14c), all five conditional cuts, every cost assumption, and the 305-window 2000-2026 extension (*t* = +3.25). The synthetic control recovers a planted effect of either sign and is silent on the null. The stamp is about the terminal-wealth gap existing — not about what causes it.
- **Tradability — Mirage.** Not eaten by costs (they cancel): there is nothing to eat. Matched for exposure, the gap is **-0.04c** with *t* = **-0.08** and a bootstrap CI of **[-1.18, +0.98]**, and the two arms earn the same reward per unit of dispersion (0.651 vs 0.608, both excess of the same cash leg). This is the desk's textbook Mirage — *it's just beta you were always paid for*: it vanishes in the 2000s (+0.67c, *t* = +0.26) and on bonds (+1.02c, CI through zero). Free and correct to act on **if** the equity weight is already chosen; not an edge to bank, scale or repeat.
- **Does DCA lower risk? — Confirmed.** Dispersion ratio **0.587**, worst window -36.13% vs -45.85%. It is bought with average exposure, not with better entry prices: the static 54.2% portfolio has the same dispersion (0.0926 vs 0.0999) and the same terminal wealth, without the schedule.